In [ ]:
#réinitialise module

%load_ext autoreload
%autoreload 2


# les différentes importations 

from IPython.display import IFrame

import random
from datetime import datetime

import sys

sys.path.append('..')
import bs4
import gtfs_kit as gk
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString
from IPython.display import HTML
import folium
from folium import plugins
import numpy as np
import branca.colormap as cm

from src.utils import (
    charger_gtfs,
    longueur_lignes,
    km_par_ligne_jour,
    km_par_ligne_plage,
    obtenir_service_ids_pour_date,
    exporter_df_to_csv,
    exporter_geojson,
    exporter_gdf_to_csv,
    
)
from src.info_reseau import dates_service, formater_date_fr, date_str, longueur_par_lignes, nom_reseau_str, chemin_logo, recuperer_logo_reseau, nom_reseau 

from src.arrets import calculer_indicateurs_arrets, afficher_statistiques
from src.cartographie import creer_carte_troncons, create_carte_arrets 
from src.create_troncons_uniques import creer_troncons_uniques
from src.indicateurs_troncons import compute_indicateurs_troncons

from src.export_html import (
    exporter_tableau_lignes_html,
    exporter_camembert_html,
    exporter_statistiques_html,
)



In [ ]:
#les chemins 


# Chemin vers le zip GTFS
BASE_DIR = os.getcwd()  # Remonte d'un niveau depuis scripts/

print(BASE_DIR)

# Choisir le jeu de données GTFS en zip, site de référence 
GTFS_ZIP_PATH = os.path.join(BASE_DIR,"data", "GTFS", "Paris_IDFM-gtfs_merge.zip")
OUTPUT_HTML_PATH = os.path.join(BASE_DIR, "output")

print(GTFS_ZIP_PATH)
print(OUTPUT_HTML_PATH)

In [ ]:

#charge GTFS en feed, longueurs lignes et nom du réseau  


    # La librairie gtfs_kit est utilisée pour charger le GTFS
feed = charger_gtfs(GTFS_ZIP_PATH)

print(type(feed))  # Vérification du type de l'objet feed

# Calcul de la longueur des shapes une seule fois, en dehors de la boucle

longueur_par_lignes=longueur_lignes(feed)

print(longueur_par_lignes)

    #cherche nom réseau 
    
# On appelle les fonctions via le module (plutôt que les noms importés
# nom_reseau_str/chemin_logo) pour ne jamais les écraser avec leur résultat :
# sinon, réexécuter cette cellule une deuxième fois lève
# "TypeError: 'str' object is not callable".
import src.info_reseau as _info_reseau

nom_reseau_str = _info_reseau.nom_reseau_str(feed)

nb_agences = len(feed.agency)
if nb_agences > 3:
    print(f"⚠ Ce GTFS regroupe {nb_agences} agences : ce que l'app ne peut pas gérer. Charger un GTFS urbain uniquement. ")


#cherche nom réseau 
chemin_logo = _info_reseau.chemin_logo(feed)

print(nom_reseau_str)

In [ ]:
# Optionnel : force un nom d'agence unique et (re-)normalise les
# route_type étendus. Utile pour un GTFS régional multi-agences (ex:
# VBB à Berlin, ~40 agences) que l'app ne peut pas gérer tel quel (max 3
# agences), ou publiant ses route_type en centaines plutôt qu'en codes
# de base — cf. src/merge_gtfs_berlin.py pour la version script.
#
# La normalisation des route_type est déjà faite automatiquement par
# charger_gtfs() (cf. src/utils.py) ; la ré-appliquer ici est sans
# risque (un code déjà de base reste inchangé) et rend l'étape visible.
#
# Passer FORCER_AGENCE_UNIQUE à False pour garder le GTFS tel quel.

FORCER_AGENCE_UNIQUE = True
NOM_VILLE_PRINCIPALE = "Nom de la ville principale"  # <- à adapter
NOM_AGENCE_FORCE = f"{NOM_VILLE_PRINCIPALE} Public Transport"

if FORCER_AGENCE_UNIQUE:
    from src.utils import normaliser_route_type_etendu

    feed.routes["route_type"] = feed.routes["route_type"].apply(normaliser_route_type_etendu)

    anciens_agency_ids = feed.agency["agency_id"].tolist()
    agency_id_unique = anciens_agency_ids[0]
    print(f"Fusion de {len(anciens_agency_ids)} agence(s) en une seule : '{NOM_AGENCE_FORCE}'")

    agence_unique = feed.agency.iloc[[0]].copy()
    agence_unique["agency_name"] = NOM_AGENCE_FORCE
    feed.agency = agence_unique.reset_index(drop=True)

    for table_nom in ("routes", "fare_attributes"):
        table = getattr(feed, table_nom)
        if table is not None and "agency_id" in table.columns:
            table["agency_id"] = agency_id_unique

    # nom_reseau_str recalculé pour refléter le nom forcé : les cellules
    # suivantes (cache, export...) s'en servent comme clé de réseau.
    import src.info_reseau as _info_reseau
    nom_reseau_str = _info_reseau.nom_fichier_valide(NOM_AGENCE_FORCE)

    print(f"nb_agences = {len(feed.agency)}, nom_reseau_str = {nom_reseau_str!r}")
    print(feed.agency)


In [ ]:

# définition plage temporelle et défintion date_JOB 

dates_service, date_debut , date_fin , date_JOB = dates_service(feed)

print(dates_service, date_debut, date_fin, date_JOB)

date_service_str, date_JOB_text = date_str(date_debut, date_fin, date_JOB)

print(date_service_str)
print(date_JOB)





In [ ]:

# Fonction pour calculer le total des kilomètres parcourus par ligne pour une journée donnée

total_vkm_per_plage=km_par_ligne_plage(dates_service,feed)

output_html_tableau=os.path.join(OUTPUT_HTML_PATH, f"tableau_ligne_plage_{nom_reseau_str}.html")

exporter_tableau_lignes_html(
    nom_reseau_str,
    date_service_str,
    feed,
    output_html_tableau,
    total_vk_plage=None,
)

HTML(filename=output_html_tableau)



In [ ]:

#export camembert avec répartition vk par mode par an 

output_html_camembert=os.path.join(OUTPUT_HTML_PATH, f"camembert_ligne_an_{nom_reseau_str}.html")

exporter_camembert_html(nom_reseau_str,date_service_str,total_vkm_per_plage, output_html_camembert)

HTML(filename=output_html_camembert)


In [ ]:
# calcul indicateurs par arrêts 

indicateurs=calculer_indicateurs_arrets(feed,date_JOB)

# Il est possible d'exporter les résultats en csv
exporter_df_to_csv(
    indicateurs,
    f"output/indicateurs_arrets_{date_JOB}_{nom_reseau_str}.csv"
)


In [ ]:
# statistique du réseau vision arrêts

output_html_statistiques=os.path.join(OUTPUT_HTML_PATH, f"statistiques_réseau_{nom_reseau_str}.html")

afficher_statistiques(indicateurs)

#exporter_statistiques_html(indicateurs, date_JOB_text, output_html_statistiques, nom_reseau_str)

exporter_statistiques_html(indicateurs, date_service_str, date_JOB_text, output_html_statistiques, nom_reseau_str)

HTML(filename=output_html_statistiques)



In [ ]:
# créer la carte des arrêts 

output_html_arret=os.path.join(OUTPUT_HTML_PATH, f"stop_maps_{nom_reseau_str}.html")

carte_arrets = create_carte_arrets(indicateurs, nom_reseau_str, date_service_str, date_JOB, GTFS_ZIP_PATH, output_html_arret, chemin_logo)


carte_arrets

In [ ]:
# Tronçons de bus
troncons_bus = creer_troncons_uniques(feed, route_type=3)
    
# Tronçons de tram
troncons_tram = creer_troncons_uniques(feed, route_type=0)

# Tronçons de metro 
troncons_metro = creer_troncons_uniques(feed, route_type=1)

# Tronçons de trolley 
troncons_trolley = creer_troncons_uniques(feed, route_type=11)

# Tronçons de ferry 
troncons_ferry = creer_troncons_uniques(feed, route_type=4)

# Tronçons de train (route_type=2 : RER, Transilien, TER...)
troncons_train = creer_troncons_uniques(feed, route_type=2)


# Il est possible d'exporter les résultats au format csv (sans géométrie, ou en geojson)
# exporter_gdf_to_csv(troncons_bus, 'output/troncons_uniques_bus.csv')
# exporter_geojson(troncons_bus, 'output/troncons_uniques_bus.geojson')

# exporter_gdf_to_csv(troncons_tram, 'output/troncons_uniques_tram.csv')
# exporter_geojson(troncons_tram, 'output/troncons_uniques_tram.geojson')

In [ ]:
# Les indicateurs sont calculés sous la forme d'un GeoDataframe par mode de transport


active_service_ids = obtenir_service_ids_pour_date(feed, date_JOB)


indicateurs_bus, indicateurs_tram, indicateurs_metro, indicateurs_trolley, indicateurs_ferry, indicateurs_train = compute_indicateurs_troncons(
    feed,  # le feed GTFS chargé plus haut
    active_service_ids,  # prise en compte uniquement des services actifs ce jour
    troncons_bus,  # Geodataframe des tronçons de bus
    troncons_tram,  # Geodataframe des tronçons de tram
    troncons_metro, # Geodataframe des tronçons de metro
    troncons_trolley, # Geodataframe des tronçons de trolley
    troncons_ferry,  # Geodataframe des tronçons de ferry
    troncons_train,  # Geodataframe des tronçons de train
)

# Il est possible d'exporter les résultats au format csv (sans géométrie, ou en geojson)
# exporter_gdf_to_csv(indicateurs_bus, f'output/indicateurs_troncons_bus_{DATE_ANALYSE}.csv')
# exporter_gdf_to_csv(indicateurs_tram, f'output/indicateurs_troncons_tram_{DATE_ANALYSE}.csv')

# exporter_geojson(indicateurs_bus, f'output/indicateurs_troncons_bus_{DATE_ANALYSE}.geojson')
# exporter_geojson(indicateurs_tram, f'output/indicateurs_troncons_tram_{DATE_ANALYSE}.geojson')

In [ ]:

# Créer la carte des tronçons

output_html_troncon=os.path.join(OUTPUT_HTML_PATH, f"troncon_maps_{nom_reseau_str}.html")

carte_troncons = creer_carte_troncons(
    indicateurs_bus,
    indicateurs_tram,
    indicateurs_metro,
    indicateurs_trolley,
    indicateurs_ferry,
    indicateurs_train,
    output_html_troncon,
    date_service_str,
    colonne_frequence='nombre_passages',
    nom_reseau_str=nom_reseau_str,
    chemin_logo=chemin_logo,
)

# Afficher la carte
carte_troncons



In [ ]:
# Pousse vers le dataset HF ww_GTFS le GTFS brut et tous les résultats déjà
# calculés dans ce notebook (dates_service, indicateurs arrêts, tronçons et
# indicateurs par mode, total_vk_plage), au même format/emplacement que le
# cache de l'app (data/memory_troncons/<réseau>/...) : l'app les retrouve
# ensuite directement sans tout recalculer.
#
# Nécessite la variable d'environnement HF_TOKEN (droits d'écriture sur
# antoinechevre/ww_GTFS), sans quoi l'envoi échoue silencieusement (les
# fichiers restent quand même écrits en local, cf. envoyer_vers_hf).

import json as _json
from src.hf_cache import envoyer_vers_hf


def pousser_resultats_vers_hf(
    nom_reseau_str,
    gtfs_zip_path,
    dates_service_liste, date_debut, date_fin, date_JOB,
    indicateurs_arrets,
    troncons_par_mode,      # dict {nom_mode: GeoDataFrame}
    indicateurs_par_mode,   # dict {nom_mode: GeoDataFrame}
    total_vk_plage,
):
    dossier_cache = os.path.join("data", "memory_troncons", nom_reseau_str)
    os.makedirs(dossier_cache, exist_ok=True)

    # dates_service.json
    chemin = os.path.join(dossier_cache, "dates_service.json")
    with open(chemin, "w") as f:
        _json.dump({
            "dates_service": dates_service_liste,
            "date_debut": date_debut,
            "date_fin": date_fin,
            "date_JOB": date_JOB,
        }, f)
    envoyer_vers_hf(chemin, f"memory_troncons/{nom_reseau_str}/dates_service.json")
    print("✓ dates_service.json")

    # indicateurs_arrets.csv
    chemin = os.path.join(dossier_cache, "indicateurs_arrets.csv")
    indicateurs_arrets.to_csv(chemin, index=False)
    envoyer_vers_hf(chemin, f"memory_troncons/{nom_reseau_str}/indicateurs_arrets.csv")
    print("✓ indicateurs_arrets.csv")

    # total_vk_plage.csv
    chemin = os.path.join(dossier_cache, "total_vk_plage.csv")
    total_vk_plage.to_csv(chemin, index=False)
    envoyer_vers_hf(chemin, f"memory_troncons/{nom_reseau_str}/total_vk_plage.csv")
    print("✓ total_vk_plage.csv")

    # troncons_<mode>.csv puis indicateurs_<mode>.csv, pour chaque mode fourni
    for nom_mode, gdf in troncons_par_mode.items():
        chemin = os.path.join(dossier_cache, f"troncons_{nom_mode.lower()}.csv")
        gdf.to_csv(chemin, index=False)
        envoyer_vers_hf(chemin, f"memory_troncons/{nom_reseau_str}/troncons_{nom_mode.lower()}.csv")
        print(f"✓ troncons_{nom_mode.lower()}.csv")

    for nom_mode, gdf in indicateurs_par_mode.items():
        chemin = os.path.join(dossier_cache, f"indicateurs_{nom_mode.lower()}.csv")
        gdf.to_csv(chemin, index=False)
        envoyer_vers_hf(chemin, f"memory_troncons/{nom_reseau_str}/indicateurs_{nom_mode.lower()}.csv")
        print(f"✓ indicateurs_{nom_mode.lower()}.csv")

    # GTFS brut
    if envoyer_vers_hf(gtfs_zip_path, f"GTFS/{os.path.basename(gtfs_zip_path)}"):
        print(f"✓ GTFS/{os.path.basename(gtfs_zip_path)}")
    else:
        print("⚠ échec envoi du GTFS brut")

    print("\n✓ Tout est poussé vers le dataset HF ww_GTFS")


In [ ]:
# Appel : adapte les dicts ci-dessous si certains modes n'ont pas été
# calculés plus haut dans le notebook (ex: pas de Train pour ce réseau).
pousser_resultats_vers_hf(
    nom_reseau_str,
    GTFS_ZIP_PATH,
    dates_service, date_debut, date_fin, date_JOB,
    indicateurs,
    {
        "Bus": troncons_bus,
        "Tram": troncons_tram,
        "Metro": troncons_metro,
        "Trolley": troncons_trolley,
        "Ferry": troncons_ferry,
        "Train": troncons_train,
    },
    {
        "Bus": indicateurs_bus,
        "Tram": indicateurs_tram,
        "Metro": indicateurs_metro,
        "Trolley": indicateurs_trolley,
        "Ferry": indicateurs_ferry,
        "Train": indicateurs_train,
    },
    total_vkm_per_plage,
)


In [ ]:
# Enregistre ce réseau dans l'index de benchmark partagé
# (benchmark/index_benchmark_reseaux.csv sur ww_GTFS), pour qu'il
# apparaisse dans l'onglet Benchmark de l'app (population en abscisse,
# indicateur transit au choix en ordonnée). Nécessite HF_TOKEN, comme
# pousser_resultats_vers_hf ci-dessus.

from src.hf_cache import fusionner_et_envoyer_csv
from src.population import population_agglomeration, deviner_ville_principale

ville_principale = deviner_ville_principale(nom_reseau_str, os.path.basename(GTFS_ZIP_PATH))
population_totale, _annee_population = population_agglomeration(ville_principale)

vk_par_mode = total_vkm_per_plage.groupby("mode")["total_km_plage"].sum()

ligne_benchmark = pd.DataFrame([{
    "reseau": nom_reseau_str,
    "ville_principale": ville_principale,
    "date_JOB": date_JOB,
    "population_totale": population_totale,
    "nombre_arrets": len(indicateurs),
    "vehicules_km_total": float(total_vkm_per_plage["total_km_plage"].sum()),
    "vehicules_km_bus": float(vk_par_mode.get("Bus", 0)),
    "vehicules_km_metro": float(vk_par_mode.get("Métro", 0)),
    "vehicules_km_tram": float(vk_par_mode.get("Tram", 0)),
}])

fusionner_et_envoyer_csv(
    ligne_benchmark,
    "benchmark/index_benchmark_reseaux.csv",
    os.path.join("data", "benchmark", "index_benchmark_reseaux.csv"),
    colonne_cle="reseau",
    valeur_cle=nom_reseau_str,
)
print(f"✓ {nom_reseau_str} enregistré dans le benchmark")
